# GeoSocialX — Quickstart (Colab)

Map the geography of **X posts**: fetch geotagged posts by radius, then find hotspots and time trends.

**The analysis half needs no API key** — this notebook runs entirely on a bundled sample, so you can try the whole pipeline in your browser with zero setup.

- PyPI: https://pypi.org/project/geosocialx/
- GitHub: https://github.com/JayeshSuryavanshi/GeoSocialX

## 1. Install

In [ ]:
!pip install -q "geosocialx[maps]"

## 2. Get a sample tweet dump

The analysis works on any newline-delimited dump of X API v2 tweets. Here we grab the bundled sample (no API needed).

In [ ]:
base = "https://raw.githubusercontent.com/JayeshSuryavanshi/GeoSocialX/main/examples"
!curl -sL {base}/sample_tweets.json -o sample_tweets.json
!curl -sL {base}/sample_places.json -o sample_places.json
print(sum(1 for _ in open("sample_tweets.json")), "sample tweets downloaded")

## 3. Load & check coverage

`coverage()` shows how sparse the geo data is — most posts aren’t geotagged.

In [ ]:
from geosocialx import GeospatialExtractor

ex = GeospatialExtractor()
tweets = ex.load_tweets("sample_tweets.json")
ex.coverage(tweets)

## 4. Extract points (incl. place resolution)

Exact-coordinate posts become points; place-only posts are resolved to their place’s bounding-box centroid. Each point records its `source` (`"exact"` or `"place"`).

In [ ]:
from collections import Counter

places = ex.load_places("sample_places.json")
points = ex.extract_points(tweets, places=places)
print(len(points), "points:", dict(Counter(p.source for p in points)))
points[0]

## 5. Analyze — where *and* when

Pure standard library: bounding box, centroid, hotspots, and temporal bins.

In [ ]:
from geosocialx import GeospatialAnalyzer

a = GeospatialAnalyzer(points)
print("summary:  ", a.summary())
print("hotspots: ", a.densest_cells(top=3))
print("time_bins:", a.time_bins("day"))

## 6. Visualize — interactive heatmap

Renders the folium map inline — drag and zoom it.

In [ ]:
import base64

from IPython.display import HTML

from geosocialx import MapVisualizer

MapVisualizer(points).to_html_map("map.html")  # needs the [maps] extra (folium)
html = open("map.html", "rb").read()
uri = "data:text/html;base64," + base64.b64encode(html).decode()
HTML(f'<iframe src="{uri}" width="100%" height="500" style="border:0"></iframe>')

## 7. Export GeoJSON

Standard library only — open it in any GIS or on [geojson.io](https://geojson.io).

In [ ]:
MapVisualizer(points).save_geojson("points.geojson")
print(open("points.geojson").read()[:300], "...")

## Fetch your own posts (optional — needs an X API token)

Everything above needs no key. To fetch live data you need an X API **bearer token** on a paid tier (Basic+); set it as `X_BEARER_TOKEN`, then:

In [ ]:
# import os
# from geosocialx import XDataFetcher
#
# fetcher = XDataFetcher(bearer_token=os.environ["X_BEARER_TOKEN"])
# tweets = fetcher.fetch_tweets("37.7749,-122.4194,10mi", count=100)  # 10 mi around SF
# fetcher.save_tweets_to_file(tweets, "tweets.json")